# Road Anomaly Detection - ML Model Training

This notebook guides you through the complete process of training a machine learning model to detect road anomalies using accelerometer data collected from the Flutter app.

## Workflow Overview
1. **Data Collection**: Export CSV files from your Flutter app
2. **Data Preprocessing**: Load, clean, and create features
3. **Model Training**: Train multiple models and compare performance
4. **Model Evaluation**: Analyze results and select best model
5. **Model Export**: Save model for deployment (TFLite or scikit-learn)

## Step 1: Setup and Installation

In [ ]:
# Install required packages (run once)
!pip install -r requirements.txt

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src directory to path
sys.path.append('src')

# Import custom modules
from data_preprocessing import load_csv_data, preprocess_data, create_windowed_features, save_processed_data
from model_training import train_random_forest, train_gradient_boosting, train_neural_network, save_model

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Step 2: Load Your Data

**Instructions:**
1. Connect your Android/iOS device to your computer
2. Navigate to the app's data directory:
   - **Android**: `/data/data/com.example.road_anomaly_collector/files/` or use `adb pull`
   - **iOS**: Use Xcode File Sharing or iTunes to export files
3. Copy all CSV files to the `data/raw/` folder
4. Run the cell below to load the data

In [ ]:
# Load all CSV files from the raw data directory
raw_data_dir = 'data/raw'

try:
    df = load_csv_data(raw_data_dir)
    print(f"\nSuccessfully loaded {len(df)} records")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("\nPlease copy your CSV files to the 'data/raw/' folder first.")
    print("See instructions above for how to export data from your Flutter app.")

## Step 3: Explore the Data

In [ ]:
# Display basic information
print("=== Dataset Info ===")
df.info()

print("\n=== First 10 Rows ===")
df.head(10)

print("\n=== Label Distribution ===")
df['Label'].value_counts()

In [ ]:
# Visualize label distribution
plt.figure(figsize=(10, 6))
df['Label'].value_counts().plot(kind='bar')
plt.title('Distribution of Road Surface Labels')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize accelerometer data sample
sample_size = min(500, len(df))
sample_df = df.iloc[:sample_size]

plt.figure(figsize=(14, 8))

plt.subplot(3, 1, 1)
plt.plot(sample_df['X'], label='X', alpha=0.7)
plt.ylabel('X Acceleration')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(sample_df['Y'], label='Y', color='orange', alpha=0.7)
plt.ylabel('Y Acceleration')
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(sample_df['Z'], label='Z', color='green', alpha=0.7)
plt.ylabel('Z Acceleration')
plt.xlabel('Sample Index')
plt.legend()

plt.suptitle('Accelerometer Data Sample')
plt.tight_layout()
plt.show()

## Step 4: Preprocess and Create Features

In [ ]:
# Preprocess the data
df_clean = preprocess_data(df.copy())

In [ ]:
# Create windowed features for ML training
# Window size: 50 samples (~1 second at 50Hz)
# Overlap: 25 samples (50% overlap for better coverage)
window_size = 50
overlap = 25

X, y = create_windowed_features(df_clean, window_size=window_size, overlap=overlap)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of features per sample: {X.shape[1]}")

## Step 5: Split Data and Train Models

In [ ]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

### Train Random Forest (Recommended starting point)

In [ ]:
# Train Random Forest classifier
rf_model, rf_metrics = train_random_forest(
    X_train, y_train, X_val, y_val,
    n_estimators=100,
    max_depth=20,
    random_state=42
)

### Train Gradient Boosting (Alternative)

In [ ]:
# Train Gradient Boosting classifier
gb_model, gb_metrics = train_gradient_boosting(
    X_train, y_train, X_val, y_val,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

### Train Neural Network (For comparison)

In [ ]:
# Get number of classes
num_classes = len(np.unique(y))
input_dim = X.shape[1]

# Train Neural Network
nn_model, nn_history = train_neural_network(
    X_train, y_train, X_val, y_val,
    input_dim=input_dim,
    num_classes=num_classes,
    epochs=50,
    batch_size=32
)

## Step 6: Compare Model Performance

In [ ]:
# Compare metrics across models
models_comparison = pd.DataFrame({
    'Random Forest': rf_metrics,
    'Gradient Boosting': gb_metrics
})

print("=== Model Comparison ===")
print(models_comparison)

# Plot comparison
plt.figure(figsize=(12, 6))
models_comparison.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xlabel('Metric')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Plot Neural Network training history
if 'nn_history' in locals():
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(nn_history['accuracy'], label='Train Acc')
    plt.plot(nn_history['val_accuracy'], label='Val Acc')
    plt.title('Neural Network Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(nn_history['loss'], label='Train Loss')
    plt.plot(nn_history['val_loss'], label='Val Loss')
    plt.title('Neural Network Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

## Step 7: Feature Importance Analysis (Random Forest)

In [ ]:
# Analyze feature importance from Random Forest
feature_names = []
stats = ['mean', 'std', 'min', 'max', 'median', 'p25', 'p75', 'var']
axes = ['X', 'Y', 'Z', 'Magnitude']

for axis in axes:
    for stat in stats:
        feature_names.append(f'{axis}_{stat}')

# Add FFT features
for axis in ['X', 'Y', 'Z']:
    feature_names.extend([f'{axis}_fft_mean', f'{axis}_fft_max', f'{axis}_fft_std'])

# Plot top 20 important features
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(10, 8))
plt.barh(range(20), importances[indices][::-1])
plt.yticks(range(20), [feature_names[i] for i in indices][::-1])
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.show()

## Step 8: Save the Best Model

In [ ]:
# Create label mapping (reverse of what was used in preprocessing)
label_mapping = {v: k for k, v in {label: idx for idx, label in enumerate(df['Label'].unique())}.items()}
print("Label mapping:", label_mapping)

# Save the best performing model (Random Forest by default)
best_model = rf_model
save_model(best_model, 'models/road_anomaly_model.pkl', label_mapping)

# Also save as TFLite compatible format if needed
print("\nModel saved successfully!")
print("You can now use this model in your Flutter app.")

## Step 9: Convert to TensorFlow Lite (Optional)

If you want to use the model directly in your Flutter app with TFLite, you'll need to convert it.

In [ ]:
# Convert scikit-learn model to ONNX, then to TFLite
# This requires additional packages: skl2onnx, onnx, tf2onnx

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    
    # Convert to ONNX
    initial_type = [('float_input', FloatTensorType([None, X.shape[1]]))]
    onnx_model = convert_sklearn(rf_model, initial_types=initial_type)
    
    # Save ONNX model
    with open("models/road_anomaly_model.onnx", "wb") as f:
        f.write(onnx_model.SerializeToString())
    
    print("Model converted to ONNX format!")
    print("Next step: Use onnx-tf or other tools to convert to TFLite")
    
except ImportError:
    print("skl2onnx not installed. Install with: pip install skl2onnx")
    print("Alternatively, train a Keras model directly for TFLite conversion.")

## Next Steps

1. **Collect More Data**: The more diverse your training data, the better your model will perform
2. **Data Augmentation**: Consider adding synthetic variations to your data
3. **Hyperparameter Tuning**: Use GridSearchCV or Optuna to optimize model parameters
4. **Cross-Validation**: Implement k-fold cross-validation for more robust evaluation
5. **Deploy to Flutter**: Integrate the trained model back into your Flutter app

## Tips for Better Results

- **Balanced Dataset**: Ensure you have roughly equal samples for each road condition
- **Quality Labels**: Make sure your manual labeling is accurate and consistent
- **Sampling Rate**: Maintain consistent sampling rate during data collection
- **Device Variation**: Collect data from multiple devices if possible
- **Environmental Factors**: Consider weather, speed, and vehicle type variations